# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIRˆ² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, available at the following URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Ensure mlcroissant is available
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and prepare to extract records.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Accessing the metadata as an object
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
We can review the available record sets, their `@id` values, and their constituent fields.

Let's list all record sets and preview their fields by unique `@id`.

In [ ]:
# List all available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s).")
for rs in record_sets:
    print(f"\nRecord Set: {rs.name}\n@id: {rs.id}\nFields:")
    for field in rs.fields:
        print(f"  - {field.name} (field @id: {field.id})")

### Example: Previewing the first few records from a record set
Let's preview records using the `@id` of the first available record set.

In [ ]:
if record_sets:
    example_record_set = record_sets[0]
    print(f"Showing records from record set: {example_record_set.name} (@id: {example_record_set.id})\n")
    for i, record in enumerate(dataset.records(record_set=example_record_set.id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Extract all data from each record set into a Pandas DataFrame using the record set and field `@id`s.

In [ ]:
# Extract data into DataFrames, keyed by record set @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records from record set: {rs.name} (@id: {rs.id})")

# Show columns (field @id) and first few records for the first record set, if available
if record_sets:
    example_rs_id = record_sets[0].id
    print(f"\nColumns in record set '{record_sets[0].name}' (@id: {example_rs_id}):")
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering and normalizing numeric fields, and grouping by categorical fields. Ensure all fields are referenced by their `@id`.


In [ ]:
# Choose a DataFrame and fields by @id for demonstration
import numpy as np

if record_sets:
    rs = record_sets[0]
    rs_id = rs.id
    df = dataframes[rs_id]

    # Attempt to pick a numeric field (by inspecting dtype or known IDs)
    # Here, we scan for the first float or int field
    numeric_fields = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_fields.append(col)
    if not numeric_fields:
        print("No numeric fields found for EDA.")
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field (by @id): {numeric_field_id}")
        
        # Set threshold for demonstration (use mean or a fixed value)
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().sum() else 0

        # Filter records with value above threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Add a normalized version for the filtered records
        if filtered_df[numeric_field_id].std() != 0 and filtered_df[numeric_field_id].notnull().sum():
            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
                filtered_df[numeric_field_id].std()
            )
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        else:
            print(f"Cannot normalize field {numeric_field_id}, possibly constant or empty.")
        # Attempt grouping by a categorical field
        group_field = None
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                group_field = col
                break
        if group_field:
            print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No record sets or data available for EDA.")

## 5. Visualization
Visualize distributions or relationships among key fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt

if record_sets and numeric_fields:
    # Plot histogram of the selected numeric field
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If there's a group field, plot mean by group
    if group_field:
        grouped_df.plot(x=group_field, y=numeric_field_id, kind='bar', legend=False, figsize=(8,4))
        plt.title(f"Mean of {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to load and explore a complex dataset described by a Croissant schema. By referencing all record sets and fields via their `@id`, we ensured reliable and reproducible extraction. Further, we performed exploratory analysis, normalization, and visualization to better understand the characteristics of the data. This workflow can be extended for more detailed domain-specific analyses, leveraging the structure and semantics provided by the Croissant standard.
